# MALAT1–A549 CRISPRi post-FlashFry reproducibility

This notebook reproduces the decision-analytic portion of the project from the
frozen 86-candidate raw decision matrix.

**It does not rebuild the GRCh38 FlashFry index.**

The output is an in-silico prioritization for experimental validation, not
experimental efficacy or safety evidence.

## 1. Locate the repository

When the project is public, paste its GitHub URL into `REPO_URL`.

Before publication, leave `REPO_URL` empty and upload the repository ZIP when
prompted.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile

REPO_URL = ""  # Example: "https://github.com/USERNAME/malat1-a549-crispri-mcda.git"
BRANCH = "main"
EXPECTED_DIR_NAME = "MALAT1_A549_CRISPRi_MCDA_GitHub_v1"

def find_project_root(start: Path):
    candidates = [start, start.parent, start.parent.parent]
    for candidate in candidates:
        if (candidate / "scripts" / "run_pipeline.py").exists():
            return candidate.resolve()
    for candidate in start.rglob("run_pipeline.py"):
        if candidate.parent.name == "scripts":
            return candidate.parent.parent.resolve()
    return None

PROJECT_ROOT = find_project_root(Path.cwd())

if PROJECT_ROOT is None and REPO_URL:
    target = Path("/content") / EXPECTED_DIR_NAME
    if target.exists():
        subprocess.run(["rm", "-rf", str(target)], check=True)
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(target)], check=True)
    PROJECT_ROOT = target.resolve()

if PROJECT_ROOT is None and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload the GitHub-ready repository ZIP.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if not zip_names:
        raise RuntimeError("No ZIP file was uploaded.")
    archive = Path("/content") / zip_names[0]
    archive.write_bytes(uploaded[zip_names[0]])
    with zipfile.ZipFile(archive) as zf:
        zf.extractall("/content")
    PROJECT_ROOT = find_project_root(Path("/content"))

if PROJECT_ROOT is None:
    raise RuntimeError(
        "Project root was not found. Set REPO_URL or run the notebook from the extracted repository."
    )

print("Project root:", PROJECT_ROOT)


## 2. Install the locked dependencies

In [ ]:
if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r",
         str(PROJECT_ROOT / "requirements-lock.txt")],
        check=True,
    )
    print("Locked dependencies installed.")
else:
    print("Local environment detected; using its existing packages.")


## 3. Run the full post-FlashFry pipeline

In [ ]:
subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "run_pipeline.py")],
    cwd=PROJECT_ROOT,
    check=True,
)
print("Pipeline finished.")


## 4. Verify the reproducibility report

In [ ]:
import json

report_path = PROJECT_ROOT / "generated" / "reproducibility_validation_report.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Validation status:", "PASS" if report["passed"] else "FAIL")
print("Numeric tolerance:", report["numeric_tolerance"])
print("Platform:", report["platform"])
assert report["passed"], "Reproducibility validation failed."


## 5. Inspect the leading candidates

In [ ]:
import csv

status_path = (
    PROJECT_ROOT / "generated" / "extended" / "tables"
    / "integrated_candidate_technical_status.tsv"
)
with status_path.open("r", encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle, delimiter="\t"))

rows = sorted(rows, key=lambda row: int(row["integrated_order"]))[:10]

headers = [
    "integrated_order",
    "candidate_id",
    "technical_tier",
    "three_profile_mean_p_top5",
    "global_weight_p_top5",
    "pareto_layer",
]
print(" | ".join(headers))
print("-" * 120)
for row in rows:
    print(" | ".join(str(row[h]) for h in headers))


## 6. Display one robustness figure

In [ ]:
from IPython.display import display, Image

figure_path = (
    PROJECT_ROOT / "generated" / "extended" / "figures"
    / "Figure_9_global_weight_smaa_top10.png"
)
if figure_path.exists():
    display(Image(filename=str(figure_path)))
else:
    print("Figure not found:", figure_path)


## Interpretation guard

A successful run demonstrates that the deposited post-FlashFry calculations
reproduce the locked scientific outputs within the stated numerical tolerance.

It does **not** demonstrate that any candidate experimentally represses MALAT1
or is biologically safe.